# 08 · RAG vs Fine-tuning vs Long Context

## Cuándo NO usar RAG

Después de varios notebooks construyendo RAGs cada vez más sofisticados, la pregunta natural es: **¿cuándo NO es la mejor opción?**

En 2026 hay tres paradigmas reales para darle conocimiento a un LLM y no hay un ganador absoluto:

- **RAG** · recupera dinámicamente N chunks por query (lo que llevamos hecho).
- **Fine-tuning** · el conocimiento queda horneado en los pesos del modelo.
- **Long context** · metes el corpus entero en cada prompt (con los modelos de contexto muy largo, del orden de cientos de miles de tokens o más).

Este notebook te da la **matriz de decisión** con 7 dimensiones, una calculadora de costes con precios reales 2026, una estimación de latencia, y tres casos prácticos resueltos para entrenar la intuición.

## Qué aprendemos

1. **Los 3 paradigmas** · qué hace cada uno y dónde brilla.
2. **Matriz de decisión** con 7 dimensiones (corpus, frecuencia, queries, coste, latencia, citación, alucinación).
3. **Calculadora de costes** con precios reales 2026 (`gpt-5-mini`, prompt caching, fine-tuning training).
4. **Break-even** RAG vs fine-tuning.
5. **Tres casos prácticos** resueltos (chatbot SaaS · asistente legal · clasificador masivo).
6. **Patrones híbridos** (RAG + FT, RAG + long context, tiered routing).

## Por qué importa

Cuando un cliente te pide "queremos un chatbot sobre nuestros documentos", la respuesta correcta no siempre es RAG. A veces fine-tuning gana, a veces long context con prompt caching es zero infra y suficiente. Esta es la conversación de ingeniería que cierra clase-4: ya sabes hacer RAG · ahora sabes decidir cuándo merece la pena.

## Conecta con

- **El resto de notebooks de este módulo (01-07 y el 09 de RAG agéntico)** · todo el RAG que ya sabes construir.
- **Módulo de Fine-tuning (clase 3)** · fine-tuning con Unsloth + Gemma 4 (el otro paradigma).
- **Módulo de Evaluación de RAG** · cómo medir si tu RAG vale lo que cuesta.
- **Módulo de coste y latencia (clase 6)** · prompt caching, model routing y batch APIs (el otro 50% del cálculo de costes).

## Stack

`Python 3.10+` · `dataclasses` · cero llamadas a LLM. **100% analítico**.

## Hardware

Cualquier laptop. No hay GPU ni descarga de modelos.

## Guion del laboratorio

1. **Los 3 paradigmas** explicados sin humo.
2. **Matriz de decisión** con 7 dimensiones.
3. **Calculadora de costes** con precios reales 2026.
4. **Break-even** entre RAG y fine-tuning por volumen.
5. **Caso A**: chatbot de soporte SaaS.
6. **Caso B**: asistente legal con jurisprudencia.
7. **Caso C**: clasificador masivo de tickets.
8. **Cuándo combinar** paradigmas.
9. **Recap y matriz visual**.

## 1. Los tres paradigmas

Antes de comparar, fijemos qué hace cada uno. Las tres opciones resuelven el **mismo problema** (darle conocimiento de dominio a un LLM) pero con **trade-offs muy distintos**.

### 1.1. RAG · Retrieval-Augmented Generation

- **Qué hace**: indexa tu corpus en un vectorstore, recupera los chunks relevantes para cada pregunta y los inyecta en el prompt.
- **Conocimiento**: vive **fuera del modelo**, en tu base vectorial. Actualizable en segundos.
- **Coste por query**: bajo. Solo metes los k chunks relevantes en el contexto (típicamente 2k-8k tokens).
- **Citación**: trivial. Sabes exactamente qué documentos usaste para responder.
- **Punto débil**: la calidad depende del **retrieval**. Si el chunk relevante no aparece en el top-k, el LLM no puede responder.

### 1.2. Fine-tuning · ajuste de pesos

- **Qué hace**: re-entrena (o ajusta con LoRA / QLoRA) los pesos del modelo sobre tus datos.
- **Conocimiento**: vive **dentro del modelo**. Horneado en los pesos.
- **Coste por query**: el más bajo de los tres. No metes contexto de dominio: el modelo ya lo sabe.
- **Citación**: imposible. No puedes decir "esto viene del doc X".
- **Punto débil**: no es actualizable sin reentrenar. Si tu corpus cambia cada semana, el fine-tuning **no es tu solución**.
- **Cuándo gana**: para enseñar **estilo, formato, dominio cerrado** o tareas de clasificación muy específicas.

### 1.3. Long context · meter todo en cada query

- **Qué hace**: con modelos de contexto muy largo, pegas tu corpus completo en el system prompt en cada llamada.
- **Conocimiento**: vive **en el contexto** de cada query. Cero indexación, cero infraestructura.
- **Coste por query**: alto. Pagas el corpus entero en input cada vez (mitigable con prompt caching, pero sigue siendo caro).
- **Citación**: posible si el modelo cita explícitamente, pero menos fiable que con RAG.
- **Punto débil**: **latencia y coste** escalan con el tamaño del corpus. Y por encima de ~500k tokens los modelos empiezan a perder precisión ("lost in the middle").
- **Cuándo gana**: corpus pequeño-mediano (como referencia, 1M tokens son unos 4 MB de texto plano), pocas queries y máxima flexibilidad sin infraestructura.

## 2. Matriz de decisión · 7 dimensiones

Esta es la tabla que pegas en tu PRD cuando te toca defender la elección ante un CTO. Para cada dimensión, qué paradigma gana:

| Dimensión | RAG | Fine-tuning | Long context |
|---|---|---|---|
| **Tamaño del corpus** | KB a TB (escalable) | MB a GB (cabe en training) | KB a pocos MB (1M tokens ≈ 4 MB de texto) |
| **Frecuencia de actualización** | Excelente: re-indexar segundos | Pésimo: re-entrenar horas o días | Excelente: cambiar el prompt |
| **Volumen de queries (qpd)** | Bueno hasta millones | Excelente a alto volumen | Malo: coste lineal con volumen |
| **Tolerancia coste por query** | Bajo (~$0.001-0.01) | Mínimo (~$0.0001-0.001) | Alto (~$0.05-1.00) |
| **Tolerancia a latencia** | Media (0.5-2s) | Baja (0.3-1s) | Alta (3-30s para 1M tokens) |
| **Necesidad de citar fuentes** | Excelente: trivial | Imposible | Posible pero frágil |
| **Riesgo de alucinaciones aceptable** | Bajo (grounded) | Medio-alto | Bajo si cabe entero |

**Regla heurística rápida**:

- Corpus **dinámico** + necesitas **citar** : RAG.
- Corpus **estático** + **alto volumen** + tarea **muy específica** : fine-tuning.
- Corpus **pequeño** + **baja frecuencia** + quieres **simplicidad** : long context.
- ¿Combinación? Casi siempre. Lo veremos en la sección 8.

## 3. Calculadora de costes · precios reales 2026

Las decisiones de paradigma se ganan o se pierden con **una hoja de cálculo**. Vamos a montar una mini-calculadora que estima el coste mensual de cada paradigma según tu caso.

Precios de referencia (`gpt-5-mini`, 2026) que vamos a usar como baseline. Ajústalos a tu proveedor real:

- **Input**: $0.25 por 1M tokens
- **Output**: $2.00 por 1M tokens
- **Embedding** (`text-embedding-3-small`): $0.02 por 1M tokens
- **Fine-tuning training**: $25 por 1M tokens de training (one-shot)
- **Fine-tuning inference**: +50% sobre input/output base

Los precios de input, output y embeddings son los publicados para `gpt-5-mini` y `text-embedding-3-small`. El prompt caching de OpenAI cobra la lectura de input cacheado a 0,10x del precio de input (por eso la calculadora usa un descuento del 90% sobre la parte cacheada). Ojo: `gpt-5-mini` tiene anunciada su retirada para diciembre de 2026, así que cuando migres de modelo actualiza también `Precios2026`. Los dos de fine-tuning son **supuestos de referencia**: cada proveedor publica su tarifa de entrenamiento e inferencia afinada por modelo, así que compruébala en su página de precios antes de usar estas cifras en una propuesta.

In [1]:
from dataclasses import dataclass


@dataclass
class Precios2026:
    """Precios de referencia de gpt-5-mini en USD por 1M tokens (ajusta a tu caso)."""
    input_per_mtok: float = 0.25
    output_per_mtok: float = 2.00
    embedding_per_mtok: float = 0.02
    ft_training_per_mtok: float = 25.00
    ft_inference_surcharge: float = 0.50  # +50% sobre input/output base


PRECIOS = Precios2026()
print(f"Precios cargados : {PRECIOS}")


Precios cargados : Precios2026(input_per_mtok=0.25, output_per_mtok=2.0, embedding_per_mtok=0.02, ft_training_per_mtok=25.0, ft_inference_surcharge=0.5)


In [2]:
def coste_rag(
    corpus_tokens: int,
    queries_mes: int,
    tokens_input_query: int = 4_000,  # k chunks recuperados + pregunta + sistema
    tokens_output_query: int = 400,
) -> dict:
    """Estima el coste mensual de un RAG.

    La indexación es one-shot (la cuento entera el mes 1). Las queries pagan
    solo los chunks recuperados, no el corpus entero.
    """
    coste_indexacion = (corpus_tokens / 1_000_000) * PRECIOS.embedding_per_mtok
    coste_input = (queries_mes * tokens_input_query / 1_000_000) * PRECIOS.input_per_mtok
    coste_output = (queries_mes * tokens_output_query / 1_000_000) * PRECIOS.output_per_mtok
    return {
        "indexacion_one_shot": round(coste_indexacion, 4),
        "queries_mes": round(coste_input + coste_output, 2),
        "total_mes_1": round(coste_indexacion + coste_input + coste_output, 2),
        "total_mes_n": round(coste_input + coste_output, 2),
    }


def coste_finetuning(
    corpus_tokens: int,
    queries_mes: int,
    tokens_input_query: int = 500,  # solo pregunta + sistema, sin contexto de dominio
    tokens_output_query: int = 400,
    epochs: int = 3,
) -> dict:
    """Estima el coste mensual de un fine-tuning.

    El training es one-shot (por versión). La inferencia es más barata porque
    el contexto de dominio ya no viaja en el prompt, pero el modelo cobra surcharge.
    """
    coste_training = (corpus_tokens * epochs / 1_000_000) * PRECIOS.ft_training_per_mtok
    surcharge = 1 + PRECIOS.ft_inference_surcharge
    coste_input = (queries_mes * tokens_input_query / 1_000_000) * PRECIOS.input_per_mtok * surcharge
    coste_output = (queries_mes * tokens_output_query / 1_000_000) * PRECIOS.output_per_mtok * surcharge
    return {
        "training_one_shot": round(coste_training, 2),
        "queries_mes": round(coste_input + coste_output, 2),
        "total_mes_1": round(coste_training + coste_input + coste_output, 2),
        "total_mes_n": round(coste_input + coste_output, 2),
    }


def coste_longcontext(
    corpus_tokens: int,
    queries_mes: int,
    tokens_output_query: int = 400,
    cache_hit_ratio: float = 0.5,  # prompt caching reduce el coste de input cacheado
    descuento_cache: float = 0.90,  # input cacheado paga solo 10% del precio
) -> dict:
    """Estima el coste mensual de meter todo el corpus en cada query.

    Aplica prompt caching: parte del corpus en cada query se cobra al 10% del precio.
    Esto modela razonablemente bien el comportamiento de Anthropic y OpenAI en 2026.
    """
    tokens_input_query = corpus_tokens + 200  # corpus completo + pregunta
    tokens_cacheados = tokens_input_query * cache_hit_ratio
    tokens_no_cacheados = tokens_input_query - tokens_cacheados
    precio_efectivo_cache = PRECIOS.input_per_mtok * (1 - descuento_cache)
    coste_input = (
        queries_mes * tokens_no_cacheados / 1_000_000 * PRECIOS.input_per_mtok
        + queries_mes * tokens_cacheados / 1_000_000 * precio_efectivo_cache
    )
    coste_output = (queries_mes * tokens_output_query / 1_000_000) * PRECIOS.output_per_mtok
    return {
        "indexacion_one_shot": 0,
        "queries_mes": round(coste_input + coste_output, 2),
        "total_mes_1": round(coste_input + coste_output, 2),
        "total_mes_n": round(coste_input + coste_output, 2),
    }


print("Funciones definidas: coste_rag, coste_finetuning, coste_longcontext")


Funciones definidas: coste_rag, coste_finetuning, coste_longcontext


### Comparativa rápida · corpus de 500k tokens, 10.000 queries/mes

Un corpus tipo "manual de empresa" (~500k tokens, equivalente a unos 500 docs medianos) con tráfico modesto.

In [3]:
CORPUS = 500_000
QUERIES = 10_000

print(f"Corpus : {CORPUS:,} tokens · Queries/mes : {QUERIES:,}\n")
print(f"RAG          : {coste_rag(CORPUS, QUERIES)}")
print(f"Fine-tuning  : {coste_finetuning(CORPUS, QUERIES)}")
print(f"Long context : {coste_longcontext(CORPUS, QUERIES)}")


Corpus : 500,000 tokens · Queries/mes : 10,000

RAG          : {'indexacion_one_shot': 0.01, 'queries_mes': 18.0, 'total_mes_1': 18.01, 'total_mes_n': 18.0}
Fine-tuning  : {'training_one_shot': 37.5, 'queries_mes': 13.88, 'total_mes_1': 51.38, 'total_mes_n': 13.88}
Long context : {'indexacion_one_shot': 0, 'queries_mes': 695.77, 'total_mes_1': 695.77, 'total_mes_n': 695.77}


**Lectura honesta de esos números**:

- **RAG** sale a unos **$18 al mes** (la indexación es 1 céntimo). Es la opción "siempre razonable" mientras el corpus no sea trivial.
- **Fine-tuning** paga **$37,5 de training** una vez y luego ~$13,9 al mes: ahorra ~$4 al mes frente a RAG, así que tarda unos **9 meses** en amortizar el training (y eso si el corpus no cambia, porque cada reentrenamiento vuelve a costar $37,5).
- **Long context** sale a **~$696 al mes**, casi 40 veces RAG, **y crece linealmente con el corpus y con las queries**. Multiplica el corpus por 4 y verás el problema.

Cambia los parámetros y juega un rato con la celda anterior. Ese es el ejercicio: **acostúmbrate a pensar en costes antes de tocar código**.

## 4. Break-even · ¿a partir de qué volumen compensa fine-tuning?

La pregunta típica: *"si voy a tener X queries/día, ¿me sale a cuenta entrenar?"*. Vamos a calcular el **punto de cruce** entre RAG y fine-tuning en función del volumen mensual de queries.

In [4]:
def break_even_rag_vs_ft(corpus_tokens: int, horizonte_meses: int = 12) -> int:
    """Devuelve el volumen mensual de queries a partir del cual fine-tuning sale
    más barato que RAG, amortizando el training en `horizonte_meses` meses.
    """
    for q in range(1_000, 100_000_000, 1_000):
        rag_total = coste_rag(corpus_tokens, q)["total_mes_n"] * horizonte_meses
        ft = coste_finetuning(corpus_tokens, q)
        ft_total = ft["training_one_shot"] + ft["queries_mes"] * horizonte_meses
        if ft_total < rag_total:
            return q
    return -1


for tam in [100_000, 500_000, 5_000_000]:
    cruce = break_even_rag_vs_ft(tam)
    if cruce > 0:
        print(f"Corpus {tam:>10,} tok : fine-tuning compensa desde ~{cruce:>9,} queries/mes")
    else:
        print(f"Corpus {tam:>10,} tok : RAG gana hasta volúmenes muy altos")


Corpus    100,000 tok : fine-tuning compensa desde ~    2,000 queries/mes
Corpus    500,000 tok : fine-tuning compensa desde ~    8,000 queries/mes
Corpus  5,000,000 tok : fine-tuning compensa desde ~   76,000 queries/mes


**Insight clave**: el break-even no está donde la intuición lo coloca. Para corpus pequeños, fine-tuning compensa pronto porque el ahorro por query es notable. Para corpus grandes, RAG aguanta mucho más porque el training es caro y, sobre todo, **el corpus cambia**.

Ojo: este cálculo **solo considera coste de inferencia**. No incluye:

- Coste de ingeniería para mantener un pipeline de fine-tuning (versionado, eval, redespliegue).
- Coste de no poder citar fuentes.
- Coste reputacional si el modelo alucina y no puedes auditar de dónde sale la respuesta.

Eso suele ser **mucho más caro** que la diferencia de tokens.

## 5. Latencia · el otro eje que se olvida

El coste no es lo único. La latencia mata productos. Estimaciones razonables 2026 con `gpt-5-mini`:

In [5]:
def latencia_estimada(tokens_input: int, tokens_output: int = 400) -> float:
    """Estima la latencia en segundos como suma de :
    - tiempo a primer token (TTFT) proporcional al input
    - tiempo de generación proporcional al output
    Valores conservadores para gpt-5-mini.
    """
    ttft = 0.3 + (tokens_input / 1_000_000) * 8  # ~8s por 1M tokens de input
    tgen = (tokens_output / 1_000) * 0.6  # ~0.6s por 1k tokens de output
    return round(ttft + tgen, 2)


casos = [
    ("RAG (4k input)", 4_000),
    ("Long context (100k input)", 100_000),
    ("Long context (500k input)", 500_000),
    ("Long context (1M input)", 1_000_000),
]
for nombre, tokens in casos:
    s = latencia_estimada(tokens)
    print(f"  {nombre:<35} : ~{s}s")


  RAG (4k input)                      : ~0.57s
  Long context (100k input)           : ~1.34s
  Long context (500k input)           : ~4.54s
  Long context (1M input)             : ~8.54s


**Regla de oro UX**: por encima de 3 segundos pierdes engagement, por encima de 10 segundos perdes al usuario. Long context con corpus grande es **inviable en interfaces conversacionales**, aunque sea "barato" en tu cálculo de coste por query.

Para procesos asíncronos (análisis batch, generación de informes) la latencia importa menos y long context se vuelve más razonable.

## 6. Caso A · chatbot de soporte SaaS

**Situación**: empresa SaaS con **500 docs** de ayuda que **cambian cada semana** (release notes, nuevos features, FAQs). Tráfico: **10.000 queries/día**. El usuario espera respuestas claras y, si es posible, con un enlace al doc.

**Datos relevantes**:
- Corpus dinámico (cambia cada semana).
- Volumen medio (300k queries/mes).
- Citación deseable (link al doc).
- Latencia exigente (chat en vivo).

In [6]:
CORPUS_A = 500_000  # ~500 docs medianos
QUERIES_A = 300_000  # 10k/día x 30

print(f"Caso A · corpus {CORPUS_A:,} tok · {QUERIES_A:,} queries/mes\n")
print(f"RAG          : {coste_rag(CORPUS_A, QUERIES_A)}")
print(f"Fine-tuning  : {coste_finetuning(CORPUS_A, QUERIES_A)}")
print(f"Long context : {coste_longcontext(CORPUS_A, QUERIES_A)}")


Caso A · corpus 500,000 tok · 300,000 queries/mes

RAG          : {'indexacion_one_shot': 0.01, 'queries_mes': 540.0, 'total_mes_1': 540.01, 'total_mes_n': 540.0}
Fine-tuning  : {'training_one_shot': 37.5, 'queries_mes': 416.25, 'total_mes_1': 453.75, 'total_mes_n': 416.25}
Long context : {'indexacion_one_shot': 0, 'queries_mes': 20873.25, 'total_mes_1': 20873.25, 'total_mes_n': 20873.25}


**Decisión: RAG**.

Razonamiento:
1. El corpus **cambia cada semana** : fine-tuning queda descartado de entrada. Cada release tendrías que re-entrenar.
2. **Quieres citar** el doc fuente : RAG lo hace nativo, fine-tuning no puede.
3. **Latencia exigente** : RAG sirve respuestas en ~1-2s, long context con todo el corpus se vuelve incómodo.
4. **Volumen alto pero no extremo** : RAG (~$540 al mes) sale unas 40 veces más barato que long context (~$20.900 al mes).
5. **Ojo con la tabla**: sobre el papel fine-tuning parece más barato (~$416 al mes de queries). Pero el corpus cambia cada semana: 4 reentrenamientos al mes son 4 × $37,5 = $150 más, y el total (~$566) ya supera a RAG, sin contar el coste de ingeniería de cada reentrenamiento.

**Posible refinamiento**: añadir un re-ranker (Cohere Rerank o similar) sobre el top-20 para mejorar precisión, y un caché por pregunta frecuente.

## 7. Caso B · asistente legal con jurisprudencia

**Situación**: bufete con **50.000 sentencias** indexadas. El corpus es **estable** (la jurisprudencia no cambia, solo se añaden sentencias nuevas mensualmente). Tráfico: **100 queries/día** de abogados. **Citar la sentencia exacta es obligatorio por compliance**.

**Datos relevantes**:
- Corpus muy grande (~100M tokens estimados).
- Estable pero incremental (no caduca, se añade).
- Bajo volumen (3.000 queries/mes).
- Citación obligatoria.
- Latencia tolerable (un abogado espera 5-10s sin problema).

In [7]:
CORPUS_B = 100_000_000  # 50k sentencias x ~2k tokens cada una
QUERIES_B = 3_000  # 100/día x 30

print(f"Caso B · corpus {CORPUS_B:,} tok · {QUERIES_B:,} queries/mes\n")
print(f"RAG          : {coste_rag(CORPUS_B, QUERIES_B, tokens_input_query=6_000)}")
print(f"Fine-tuning  : {coste_finetuning(CORPUS_B, QUERIES_B)}")
# long context aquí ni cabe (100M > 1M tokens), solo lo mostramos como referencia teórica
print(f"Long context : NO CABE en ventana de contexto (corpus 100x más grande que 1M tokens)")


Caso B · corpus 100,000,000 tok · 3,000 queries/mes

RAG          : {'indexacion_one_shot': 2.0, 'queries_mes': 6.9, 'total_mes_1': 8.9, 'total_mes_n': 6.9}
Fine-tuning  : {'training_one_shot': 7500.0, 'queries_mes': 4.16, 'total_mes_1': 7504.16, 'total_mes_n': 4.16}
Long context : NO CABE en ventana de contexto (corpus 100x más grande que 1M tokens)


**Decisión: RAG**.

Razonamiento:
1. **Long context queda fuera** físicamente : 100M tokens no caben en ninguna ventana 2026.
2. **Fine-tuning** tendría coste de training enorme (~$7.500), no permite citar la sentencia exacta y no es actualizable cuando entren nuevas sentencias cada mes. Descartado por compliance.
3. **RAG** con re-ranker y citaciones estructuradas es **el estándar de la industria legal** en 2026.

**Posible refinamiento**: hibridar BM25 + embeddings (las búsquedas legales se benefician mucho de coincidencia léxica de artículos y términos técnicos), filtros por jurisdicción y por fecha, y un evaluador LLM-as-judge que verifique que la sentencia citada realmente respalda la afirmación.

## 8. Caso C · clasificador masivo de tickets

**Situación**: agente que clasifica **1M tickets/día** en 12 categorías predefinidas. No genera texto: solo emite una etiqueta. Las categorías **no cambian** (taxonomía estable). Hay **dataset histórico de 100k tickets etiquetados** por humanos.

**Datos relevantes**:
- Tarea muy específica y acotada (clasificación cerrada).
- Volumen extremo (30M queries/mes).
- Sin necesidad de citar.
- Latencia : importa la throughput total, no la individual.
- Dataset etiquetado disponible.

In [8]:
CORPUS_C = 50_000_000  # 100k tickets x ~500 tokens cada uno como training set
QUERIES_C = 30_000_000  # 1M/día x 30

print(f"Caso C · corpus training {CORPUS_C:,} tok · {QUERIES_C:,} queries/mes\n")
# para clasificación el output es muy corto (1 token de etiqueta)
rag_c = coste_rag(CORPUS_C, QUERIES_C, tokens_input_query=2_000, tokens_output_query=10)
ft_c = coste_finetuning(CORPUS_C, QUERIES_C, tokens_input_query=300, tokens_output_query=10)
print(f"RAG          : {rag_c}")
print(f"Fine-tuning  : {ft_c}")
print(f"Long context : inviable a 30M queries/mes con cualquier corpus relevante")


Caso C · corpus training 50,000,000 tok · 30,000,000 queries/mes

RAG          : {'indexacion_one_shot': 1.0, 'queries_mes': 15600.0, 'total_mes_1': 15601.0, 'total_mes_n': 15600.0}
Fine-tuning  : {'training_one_shot': 3750.0, 'queries_mes': 4275.0, 'total_mes_1': 8025.0, 'total_mes_n': 4275.0}
Long context : inviable a 30M queries/mes con cualquier corpus relevante


**Decisión: fine-tuning**.

Razonamiento:
1. **Volumen extremo** : a 30M queries/mes cada token de prompt es dinero. Eliminar el contexto de dominio del prompt (gracias al fine-tuning) ahorra una fortuna.
2. **Tarea cerrada y estable** : 12 categorías que no cambian. Justo el escenario donde el modelo "se aprende" la tarea bien.
3. **Dataset etiquetado existe** : tienes los 100k ejemplos para entrenar.
4. **No necesitas citar** : es clasificación, no QA.
5. **Latencia individual irrelevante** : importa el throughput, y un modelo más pequeño afinado va más rápido.

**Posible refinamiento**: empezar con un modelo más pequeño (Gemma 4 o Llama 4 small) afinado con QLoRA, y caer a un modelo grande solo para tickets con baja confianza (`< 0.7`). Esto se llama **tiered routing** y es el patrón productivo de 2026 para clasificación masiva.

## 9. Cuándo combinar paradigmas

En producción real **casi nadie elige un solo paradigma**. Los patrones híbridos de 2026:

### 9.1. RAG + fine-tuning

El más común. Fine-tuneas el modelo para **el estilo y formato** de tu dominio (tono de soporte, estructura de respuesta JSON, terminología interna) y usas RAG para **el conocimiento dinámico**.

- Ejemplo : asistente médico fine-tuneado para responder con estructura clínica + RAG sobre la base de protocolos actualizada cada semana.
- Beneficio : combinas la **consistencia** del fine-tuning con la **frescura** del RAG.

### 9.2. RAG + long context

Cuando los chunks individuales son **demasiado pequeños** para razonar y necesitas más contexto. El retriever filtra a top-50 docs, pero los **metes todos** (no solo top-3) en una ventana de 1M tokens.

- Ejemplo : análisis de un caso legal donde necesitas leer las 50 sentencias más parecidas enteras, no solo párrafos.
- Beneficio : retrieval barato y precisión alta sin sacrificar contexto.

### 9.3. Long context + caché de prompt

Si tu corpus es **pequeño y estable** (manuales de producto, documentación de API), meterlo entero en el system prompt con **prompt caching** lo deja casi gratis. Es la opción "cero infraestructura".

- Ejemplo : asistente que responde sobre un libro de 80k tokens. Cache el libro entero, pagas solo las preguntas.
- Beneficio : zero indexación, zero vectorstore, mantenimiento mínimo.

### 9.4. Tiered routing

Para alto volumen, primer paso con modelo pequeño fine-tuneado (rápido, barato), escalado a modelo grande con RAG solo si confianza baja.

- Ejemplo : el 90% de queries de soporte se resuelven con un Gemma 4 fine-tuneado; el 10% complejo va a `gpt-5-mini` + RAG.
- Beneficio : coste mínimo manteniendo techo de calidad.

## 10. Recap · matriz visual de decisión

Para cerrar, una vista compacta de "si tu caso es **X**, elige **Y**":

| Tu caso es... | Tu paradigma es... |
|---|---|
| Corpus que cambia cada semana + necesitas citar | **RAG** |
| Corpus muy grande (>1M tokens) | **RAG** (long context no cabe) |
| Tarea cerrada, alto volumen, sin citación | **Fine-tuning** |
| Corpus pequeño y estable, pocas queries | **Long context** (zero infra) |
| Estilo de respuesta crítico + corpus dinámico | **RAG + fine-tuning** |
| Necesitas razonar sobre 50 docs enteros | **RAG + long context** |
| 90% queries triviales, 10% complejas | **Tiered routing** |

### Las tres ideas que llevarte

1. **No hay un ganador absoluto**. Hay un ganador para tu caso, con tu corpus, tu volumen, tu tolerancia a coste y latencia, y tus requisitos de citación.
2. **Coste no es solo el coste por query**. Mantener un pipeline de fine-tuning, no poder citar fuentes o aceptar 10s de latencia tienen un coste oculto enorme.
3. **La elección casi nunca es pura**. Los sistemas de producción combinan paradigmas. RAG sigue siendo el ancla en 2026 porque resuelve el problema más difícil: **conocimiento dinámico con auditabilidad**.

### Retos para practicar

1. **Modifica `Precios2026`** con los precios reales de tu proveedor actual (Anthropic, OpenAI, Google) y vuelve a correr la sección 4. ¿Cambia el break-even?
2. **Inventa un caso D** : una empresa de tu sector con datos realistas (corpus, queries, frescura) y decide el paradigma justificándolo con los costes calculados.
3. **Implementa tiered routing** en pseudocódigo : una función `responder(pregunta)` que decida si usar Gemma fine-tuneado o `gpt-5-mini` + RAG según un score de confianza.